# Live slate

Every game on the newest Pinnacle board. The rotation, shot, and event models are fit once, on completed games before that slate date. Each game then gets its own script: that game's spread, total, posted five, and outs. `BOOK` prices the overs. `N_SIMS` is the number of rows per game. The seed is the matchup, so one game does not depend on another.

Prop probabilities, correlations, expected value, and portfolio risk are the four tables below the run. A correlation is two overs in the same game. A portfolio is one dollar on each of that game's five highest overs. Games do not share rows, so the day book adds those game portfolios as independent.


In [1]:
import json
import re
import sys
import zlib
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "nba_game_simulator").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("open this notebook from the nba_quant_analysis repo")

REPO = repo_root()
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "nba_game_simulator" / "src"))

from game_boxscore import (
    Prop,
    assemble_box_score,
    portfolio_risk,
    price_over,
    prop_probability,
    stat_values,
)
from game_enviroment import (
    build_features,
    build_game_table,
    load_gamelogs,
    load_model,
    player_slate,
    print_sim_summary,
    save_model,
    season_of,
    simulate_game,
    train,
)
from game_opportunities import (
    allocate_game_opportunities,
    fit_opportunities,
    live_opportunities,
    training_opportunities,
)
from game_rotations import (
    allocate_rotation,
    fit_rotation,
    invalid_segment_count,
    play_rates,
    script_exposure,
    script_rates,
    segments_from_minutes,
    training_minutes,
)
from lineup import (
    attach_lineup_fill,
    build_lineup_features,
    fill_lineup_gaps,
    merge_lineup_features,
)
from role_context import attach_script, role_for_slate
from shared_events import allocate_events, fit_events, live_events, training_events
from src.live.predict_points import lineups_on, probable_on, unavailable_on
from src.live.prop_markets import attach_roster, book_names, latest_prop_markets
from src.pipeline.silver.rotowire import normalize_team_abbreviation
from src.scrapers.arbitrage import canonical_team

BOOK = "underdog"
N_SIMS = 10000
MODEL_PATH = REPO / "nba_game_simulator" / "game_env_model.json"
PINNACLE_DIR = REPO / "data" / "props" / "pinnacle" / "nba"
LINEUP_DIR = REPO / "data" / "props" / "rotowire_nba_lineup"
STAT_LABELS = (
    ("PTS", "pts"),
    ("REB", "reb"),
    ("AST", "ast"),
    ("STL", "stl"),
    ("BLK", "blk"),
    ("PA", "p+a"),
    ("PRA", "p+r+a"),
    ("RA", "r+a"),
)


In [2]:
def newest(folder: Path, pattern: str) -> Path:
    files = sorted(path for path in folder.glob(pattern) if path.is_file())
    if not files:
        raise FileNotFoundError(f"no {pattern} in {folder}")
    return files[-1]


def snapshot_date(path: Path) -> str:
    match = re.search(r"\d{4}-\d{2}-\d{2}", path.name)
    if match is None:
        raise ValueError(f"no slate date in {path.name}")
    return match.group(0)


def abbr_for(label: str) -> str:
    wanted = canonical_team(label, "nba")
    for abbr in roster["team_abbreviation"].dropna().unique():
        normalized = normalize_team_abbreviation(abbr)
        if canonical_team(normalized, "nba") == wanted:
            return normalized
    return label


log_paths = sorted((REPO / "data" / "silver" / "nba").glob("*/regular_season/player_gamelogs.parquet"))
if not log_paths:
    raise FileNotFoundError("no player gamelogs under data/silver/nba")
roster_path = log_paths[-1]
played = pd.read_parquet(
    roster_path,
    columns=["player_name", "player_id", "team_abbreviation", "game_date"],
)
played["game_date"] = pd.to_datetime(played["game_date"])
roster = (
    played.sort_values(["player_id", "game_date"])
    .groupby("player_id", as_index=False)
    .tail(1)[["player_name", "player_id", "team_abbreviation"]]
)

odds_path = newest(PINNACLE_DIR, "pinnacle_nba_*_team.json")
lineup_path = newest(LINEUP_DIR, "rotowire_nba_*.json")
slate_date = snapshot_date(odds_path)
if snapshot_date(lineup_path) != slate_date:
    raise ValueError(
        f"odds are {slate_date} and lineups are {snapshot_date(lineup_path)}"
    )
board = json.loads(odds_path.read_text())["games"]
posted = lineups_on(slate_date)
outs = unavailable_on(slate_date)
probable = probable_on(slate_date)

games = []
for game in board:
    participants = game.get("participants") or []
    if len(participants) < 2:
        continue
    away_name, home_name = participants[0], participants[1]
    markets = game.get("team_markets") or {}
    total = None
    for block in markets.get("total") or []:
        if block.get("period") not in (0, None) or block.get("is_alternate"):
            continue
        for line in block.get("lines") or []:
            if line.get("side") == "over" and line.get("points") is not None:
                total = line["points"]
                break
    home_spread = None
    for block in markets.get("spread") or []:
        if block.get("period") not in (0, None) or block.get("is_alternate"):
            continue
        for line in block.get("lines") or []:
            if line.get("side") == "home" and line.get("points") is not None:
                home_spread = line["points"]
                break
    home = abbr_for(home_name)
    away = abbr_for(away_name)
    games.append(
        {
            "away": away,
            "home": home,
            "home_spread": home_spread,
            "total": total,
            "away_five": " ".join(posted.get(away) or []) or "—",
            "home_five": " ".join(posted.get(home) or []) or "—",
        }
    )
if not games:
    raise ValueError(f"no games on {odds_path.name}")

print(f"Pinnacle: {odds_path.name}")
print(f"Rotowire: {lineup_path.name}")
print(f"roster:   {roster_path.relative_to(REPO)} through {played['game_date'].max().date()}")
display(pd.DataFrame(games))


Pinnacle: pinnacle_nba_2026-11-08_123000_team.json
Rotowire: rotowire_nba_2026-11-08_123000.json
roster:   data/silver/nba/2026-27/regular_season/player_gamelogs.parquet through 2026-11-07


,away,home,home_spread,total,away_five,home_five
0,BOS,MIL,12.5,222.5,Derrick White Hugo Gonzalez Jayson Tatum Paul ...,Brayden Burries Tyler Herro Jaime Jaquez Kyle ...
1,BKN,DAL,-7.5,224.0,Nolan Traore Michael Porter Danny Wolf Julius ...,Kyrie Irving Max Christie Cooper Flagg P.J. Wa...
2,LAL,DET,-7.0,217.0,Austin Reaves Quentin Grimes Ziaire Williams M...,Cade Cunningham Duncan Robinson Ausar Thompson...
3,MIA,SAC,10.0,238.0,Davion Mitchell Klay Thompson Andrew Wiggins G...,Darius Acuff Zach LaVine De'Andre Hunter Keega...
4,NOP,LAC,1.5,226.5,Dejounte Murray Jeremiah Fears Trey Murphy Zio...,Darius Garland Kris Dunn Derrick Jones Rui Hac...
5,PHI,CHI,9.5,239.0,Tyrese Maxey VJ Edgecombe Jaylen Brown LeBron ...,Josh Giddey Isaac Okoro Matas Buzelis Caleb Wi...
6,UTA,MIN,-12.0,223.0,Keyonte George Darryn Peterson Ace Bailey Laur...,LaMelo Ball Ayo Dosunmu Jonathan Kuminga Jaden...


In [3]:
import argparse

saved = json.loads(MODEL_PATH.read_text()) if MODEL_PATH.exists() else None
if saved is not None and saved.get("version") == 3:
    model = load_model(MODEL_PATH)
    print(f"loaded {MODEL_PATH.name}  ({model.get('trained_games')} games)")
else:
    if saved is not None:
        print(
            f"{MODEL_PATH.name} is version {saved.get('version')}; "
            "retraining version 3"
        )
    raw = load_gamelogs(
        str(REPO / "data" / "silver" / "nba" / "*" / "regular_season" / "player_gamelogs.parquet")
    )
    env_games = build_game_table(raw)
    lineups, lineup_state = build_lineup_features(
        raw, halflife=12.0, prior_weight=4.0, season_carry=0.6, window=300,
    )
    env_games = merge_lineup_features(env_games, lineups)
    lineup_state = attach_lineup_fill(lineup_state, env_games)
    env_games = fill_lineup_gaps(env_games)
    env_games, state = build_features(
        env_games, halflife=12.0, prior_w=4.0, carry=0.6, window=300, min_league=30,
    )
    state["lineup"] = lineup_state
    args = argparse.Namespace(
        test_seasons=1,
        validation_season="2025-26",
        holdout_season="2026-27",
        halflife=12.0,
        prior_weight=4.0,
        season_carry=0.6,
        league_window=300,
        spread_convention="auto",
        plots=False,
    )
    model = train(env_games, state, args)
    save_model(model, str(MODEL_PATH))


loaded game_env_model.json  (8367 games)


In [4]:
ROLE_COLUMNS = [
    "season_year",
    "player_id",
    "player_name",
    "team_id",
    "team_abbreviation",
    "game_id",
    "game_date",
    "matchup",
    "min",
    "pf",
    "team_pace",
    "team_off_rating",
    "team_def_rating",
    "pace",
    "off_rating",
    "def_rating",
    "start_position",
    "pos",
    "team_pts",
    "opp_pts",
    "player_team_spread",
    "fga",
    "fgm",
    "fg3_a",
    "fg3_m",
    "fta",
    "ftm",
    "tov",
    "team_fga",
    "team_poss",
    "team_tov",
    "ast",
    "oreb",
    "dreb",
    "stl",
    "blk",
    "team_fgm",
    "team_ast",
    "team_oreb",
    "team_stl",
    "team_blk",
    "opp_dreb",
    "opp_tov",
    "opp_fga",
]
logs = pd.concat(
    [pd.read_parquet(path, columns=ROLE_COLUMNS) for path in log_paths],
    ignore_index=True,
)
logs["game_date"] = pd.to_datetime(logs["game_date"])
slate_day = pd.Timestamp(slate_date)
history = logs.loc[logs["game_date"] < slate_day]
if history.empty:
    raise ValueError(f"no completed games before {slate_date}")
this_season = history.loc[history["season_year"].astype(str).eq(season_of(slate_day))]
latest = (
    this_season.sort_values(["player_id", "game_date", "game_id"])
    .groupby("player_id", as_index=False)
    .tail(1)
)
latest["abbr"] = latest["team_abbreviation"].map(normalize_team_abbreviation)

trained = training_minutes(history)
rotation = fit_rotation(trained)
shot_rows = training_opportunities(history)
shot_model = fit_opportunities(history)
event_rows = training_events(history)
event_model = fit_events(history)
pbp_all = pd.read_parquet(REPO / "data" / "PlayByPlay" / "s27_pbp.parquet")
print(
    f"rotation fit on {len(trained):,} player-games"
    f" through {history['game_date'].max().date()}"
)
print(
    f"shot model fit on {shot_model.metadata['player_games']:,} player-games"
    f" through {shot_model.training_cutoff}"
    f"  ({len(shot_rows):,} rolling rows)"
)
print(
    f"event model fit through {event_model.cutoff}"
    f"  ({len(event_rows):,} rolling rows)"
)

markets, market_audit = latest_prop_markets(REPO / "data" / "props", BOOK)
others = ", ".join(name for name in book_names() if name != market_audit["book"])
print(
    f"{market_audit['book']} from {market_audit['source_file']}. "
    f"Change BOOK to {others}."
)
if market_audit["book"] == "prizepicks":
    print("PrizePicks has no two-way price. Both sides are -119.")


/Users/alexgonzalez/Documents/nba_quant_analysis/nba_game_simulator/src/game_opportunities/train.py:38: UserWarning: zero-minute rows have positive statistics; per-minute denominators exclude them
  frame = _prepare_opportunity_logs(logs)
/Users/alexgonzalez/Documents/nba_quant_analysis/nba_game_simulator/src/game_opportunities/train.py:82: UserWarning: zero-minute rows have positive statistics; per-minute denominators exclude them
  frame = _prepare_opportunity_logs(logs)


rotation fit on 179,871 player-games through 2026-11-07
shot model fit on 179,871 player-games through 2026-11-07  (179,871 rolling rows)
event model fit through 2026-11-07  (179,871 rolling rows)
underdog from underdog_nba_2026-11-08_123004.json. Change BOOK to prizepicks, novig, prophetx, pinnacle, fanduel, draftkings.


In [5]:
def team_id_for(abbr: str) -> int:
    ids = latest.loc[latest["abbr"].eq(abbr), "team_id"].dropna()
    if ids.empty:
        raise ValueError(f"no completed games for {abbr} before {slate_date}")
    return int(ids.mode().iloc[0])


def players_for(abbr, own_id, other_id, is_home, spread, starter_ids, out_ids, probable_ids):
    starters = {int(player_id) for player_id in (starter_ids or [])}
    out_ids = {int(player_id) for player_id in (out_ids or [])}
    probable_ids = {int(player_id) for player_id in (probable_ids or [])}
    side = latest.loc[latest["abbr"].eq(abbr)]
    posted = latest.loc[latest["player_id"].astype(int).isin(starters | out_ids | probable_ids)]
    missing = (starters | out_ids | probable_ids).difference(posted["player_id"].astype(int))
    if missing:
        print(f"{abbr} posted players missing from logs: {sorted(missing)}")
    side = pd.concat([side, posted], ignore_index=True).drop_duplicates("player_id")
    if side.empty:
        raise ValueError(f"no players for {abbr} before {slate_date}")
    on_roster = side["player_id"].astype(int)
    return pd.DataFrame(
        {
            "player_id": side["player_id"].to_numpy(),
            "player_name": side["player_name"].to_numpy(),
            "team": abbr,
            "game_date": slate_day,
            "team_id": own_id,
            "opp_team_id": other_id,
            "is_home": float(bool(is_home)),
            "is_starter": on_roster.isin(starters).astype(float).to_numpy(),
            "probable": on_roster.isin(probable_ids).astype(float).to_numpy(),
            "available": (~on_roster.isin(out_ids)).astype(float).to_numpy(),
            "team_spread": float(spread),
        }
    )


def check_minutes(label, environment, side, drawn, segments):
    row = environment.iloc[0]
    invalid = invalid_segment_count(
        segments[0],
        side,
        game_minutes=float(row["game_minutes"]),
        n_ot=float(row["n_ot"]),
    )
    if invalid != 0:
        raise AssertionError(f"{label} script 0 has {invalid} invalid segments")
    budget = 5.0 * environment["game_minutes"].to_numpy()
    gap = np.max(np.abs(drawn.sum(axis=1) - budget))
    if gap > 1e-4:
        raise AssertionError(f"{label} minutes miss the script budget by {gap:.4f}")
    if np.any(drawn > environment["game_minutes"].to_numpy()[:, None] + 1e-6):
        raise AssertionError(f"{label} has a player above the game length")


def ordered_profile(profile, side, label):
    ordered = profile.set_index("player_id").loc[side["player_id"]].reset_index()
    if len(ordered) != len(side) or not ordered["player_id"].equals(side["player_id"]):
        raise AssertionError(f"{label} profile does not match the rotation")
    return ordered


def run_game(slate):
    team_abbr = normalize_team_abbreviation(slate["team"])
    opp_abbr = normalize_team_abbreviation(slate["opponent"])
    team_id = team_id_for(team_abbr)
    opp_id = team_id_for(opp_abbr)
    players = pd.concat(
        [
            players_for(
                team_abbr,
                team_id,
                opp_id,
                slate["is_home"],
                slate["spread"],
                slate["team_lineup"],
                slate["team_out"],
                slate["team_probable"],
            ),
            players_for(
                opp_abbr,
                opp_id,
                team_id,
                not slate["is_home"],
                -float(slate["spread"]),
                slate["opponent_lineup"],
                slate["opponent_out"],
                slate["opponent_probable"],
            ),
        ],
        ignore_index=True,
    )
    names = players[["player_id", "player_name", "team", "available"]]
    role = role_for_slate(logs, players.drop(columns=["player_name", "team"]))
    role = names.merge(role, on="player_id", how="left", validate="one_to_one")
    role = role.merge(
        players[["player_id", "probable"]],
        on="player_id",
        how="left",
        validate="one_to_one",
    )
    role["side"] = role["team"].eq(team_abbr).map({True: 0, False: 1})
    role = role.sort_values(
        ["side", "is_starter", "min_mean_10"],
        ascending=[True, False, False],
    ).drop(columns="side")
    rates = play_rates(history, role)
    role["play_rate"] = rates["play_rate"].to_numpy()
    recent_ids = []
    for side_id in role["team_id"].drop_duplicates():
        side_games = (
            history.loc[history["team_id"].eq(side_id), ["game_id", "game_date"]]
            .drop_duplicates("game_id")
            .sort_values("game_date")
            .tail(12)["game_id"]
        )
        recent_ids.extend(side_games.astype(str))
    recent_ids = set(recent_ids)
    exposure_logs = history.loc[
        history["game_id"].astype(str).isin(recent_ids),
        ["game_id", "player_id", "team_id", "game_date", "min", "player_name", "start_position"],
    ]
    pbp = pbp_all.loc[pbp_all["gameId"].astype(str).isin(recent_ids)]
    script = script_rates(script_exposure(pbp, exposure_logs), role)
    role["close_rate"] = script["close_rate"].to_numpy()
    role["blowout_rate"] = script["blowout_rate"].to_numpy()
    team_role = role.loc[role["team"].eq(team_abbr)].reset_index(drop=True)
    opp_role = role.loc[role["team"].eq(opp_abbr)].reset_index(drop=True)

    seed = zlib.crc32(f"{slate_date}|{opp_abbr}|{team_abbr}".encode())
    sim = simulate_game(
        model,
        slate["team"],
        slate["opponent"],
        slate["date"],
        total=slate["total"],
        spread=slate["spread"],
        is_home=bool(slate["is_home"]),
        team_lineup=slate["team_lineup"],
        opponent_lineup=slate["opponent_lineup"],
        n=N_SIMS,
        seed=int(seed),
    )
    print_sim_summary(sim, slate["total"], slate["spread"])
    opp_env = sim.copy()
    opp_env["team_margin"] = -opp_env["team_margin"]
    opp_env["is_home"] = 1.0 - opp_env["is_home"].astype(float)
    game_key = f"{slate_date}-{team_abbr}-{opp_abbr}"
    for frame in (sim, opp_env):
        frame["game_id"] = game_key
        frame["game_date"] = slate_day
    rng = np.random.default_rng(int(seed))
    minutes = {
        team_abbr: allocate_rotation(rotation, team_role, sim, rng, method="segments"),
        opp_abbr: allocate_rotation(rotation, opp_role, opp_env, rng, method="segments"),
    }
    schedules = {
        team_abbr: segments_from_minutes(team_role, sim, minutes[team_abbr]),
        opp_abbr: segments_from_minutes(opp_role, opp_env, minutes[opp_abbr]),
    }
    check_minutes(team_abbr, sim, team_role, minutes[team_abbr], schedules[team_abbr])
    check_minutes(opp_abbr, opp_env, opp_role, minutes[opp_abbr], schedules[opp_abbr])

    shot_profile = live_opportunities(history, players.drop(columns=["player_name", "team"]))
    shot_profile["player_id"] = shot_profile["player_id"].astype(players["player_id"].dtype)
    shot_profile = names.drop(columns="available").merge(
        shot_profile, on="player_id", how="left", validate="one_to_one"
    )
    if sim["is_home"].nunique() != 1:
        raise AssertionError("the slate does not have one home flag")
    slate_is_home = bool(sim["is_home"].iloc[0])
    home_abbr, away_abbr = (team_abbr, opp_abbr) if slate_is_home else (opp_abbr, team_abbr)
    home_role, away_role = (team_role, opp_role) if slate_is_home else (opp_role, team_role)
    game = f"{away_abbr} @ {home_abbr}"
    allocated = allocate_game_opportunities(
        shot_model,
        ordered_profile(shot_profile, home_role, "shot"),
        ordered_profile(shot_profile, away_role, "shot"),
        minutes[home_abbr],
        minutes[away_abbr],
        sim,
        rng,
        home_segments=schedules[home_abbr],
        away_segments=schedules[away_abbr],
    )
    shots = {home_abbr: allocated["home"], away_abbr: allocated["away"]}
    for drawn in shots.values():
        if not drawn["metadata"]["segments_used"]:
            raise AssertionError("shots were not kept inside the Layer 3 segments")
    for label, scenario in ((team_abbr, sim["team_pts"]), (opp_abbr, sim["opp_pts"])):
        target = np.floor(scenario.to_numpy(dtype=float) + 0.5).astype(np.int64)
        if not np.array_equal(shots[label]["PTS"].sum(axis=1), target):
            raise AssertionError(f"{label} points miss the scenario score")
    for label, side in ((team_abbr, team_role), (opp_abbr, opp_role)):
        unavailable = side["available"].to_numpy() == 0
        if unavailable.any() and shots[label]["FGA"][:, unavailable].sum() != 0:
            raise AssertionError(f"{label} gave a shot to an unavailable player")

    event_profile = live_events(history, players.drop(columns=["player_name", "team"]))
    event_profile["player_id"] = event_profile["player_id"].astype(players["player_id"].dtype)
    event_profile = names.drop(columns="available").merge(
        event_profile, on="player_id", how="left", validate="one_to_one"
    )
    availability = {
        team_abbr: team_role["available"].to_numpy() == 1,
        opp_abbr: opp_role["available"].to_numpy() == 1,
    }
    events, event_details = allocate_events(
        event_model,
        {
            team_abbr: ordered_profile(event_profile, team_role, "event"),
            opp_abbr: ordered_profile(event_profile, opp_role, "event"),
        },
        shots,
        minutes,
        rng,
        segments=schedules,
        availability=availability,
        return_details=True,
    )
    if not event_details["segments_used"]:
        raise AssertionError("events were not kept inside the Layer 3 segments")
    for label, side in ((team_abbr, team_role), (opp_abbr, opp_role)):
        if event_details["sources"][label]["AST"] != "layer4":
            raise AssertionError(f"{label} assists were drawn again")
        if event_details["sources"][label]["OREB"] != "layer4":
            raise AssertionError(f"{label} offensive rebounds were drawn again")
        if not np.array_equal(events[label]["AST"], shots[label]["AST"]):
            raise AssertionError(f"{label} assists are not the Layer 4 assists")
        if not np.array_equal(events[label]["REB"], events[label]["OREB"] + events[label]["DREB"]):
            raise AssertionError(f"{label} rebounds are not offensive plus defensive")
        unavailable = side["available"].to_numpy() == 0
        if unavailable.any():
            for stat in ("AST", "OREB", "DREB", "STL", "BLK"):
                if events[label][stat][:, unavailable].sum() != 0:
                    raise AssertionError(f"{label} gave {stat} to an unavailable player")
    for label, other in ((team_abbr, opp_abbr), (opp_abbr, team_abbr)):
        misses = shots[label]["FGA"].sum(axis=1) - shots[label]["FGM"].sum(axis=1)
        rebounds = events[label]["OREB"].sum(axis=1) + events[other]["DREB"].sum(axis=1)
        if np.any(rebounds > misses):
            raise AssertionError(f"rebounds on {label} misses exceed those misses")
        if np.any(events[other]["STL"].sum(axis=1) > shots[label]["TOV"].sum(axis=1)):
            raise AssertionError(f"steals exceed {label} turnovers")
        if np.any(events[other]["BLK"].sum(axis=1) > misses):
            raise AssertionError(f"blocks on {label} shots exceed its misses")

    box = assemble_box_score(
        sim,
        {"home": home_role["player_id"], "away": away_role["player_id"]},
        {"home": minutes[home_abbr], "away": minutes[away_abbr]},
        {"home": shots[home_abbr], "away": shots[away_abbr]},
        {"home": events[home_abbr], "away": events[away_abbr]},
        check_score=False,
    )
    if box.n_sims != len(sim):
        raise AssertionError("the box score does not have one row per script")

    game_roster = pd.concat(
        [
            home_role.assign(team=home_abbr, side="home"),
            away_role.assign(team=away_abbr, side="away"),
        ],
        ignore_index=True,
    )
    offered = attach_roster(
        markets,
        game_roster.loc[:, ["team", "side", "player_id", "player_name"]],
    )
    slate_dates = sorted(set(pd.to_datetime(offered["slate_date"]).dt.date)) if not offered.empty else []
    if offered.empty:
        print(f"{game}: no {market_audit['book']} lines matched")
    elif slate_dates != [slate_day.date()]:
        print(f"{game}: market dates {slate_dates}; the slate date is {slate_day.date()}")

    ranked_rows = []
    for stat, label in STAT_LABELS:
        for offer in offered.loc[offered["stat"].eq(stat)].itertuples(index=False):
            index = list(box.teams[offer.side].player_ids).index(offer.player_id)
            column = stat_values(box, offer.side, stat)[:, index]
            priced = price_over(
                box,
                Prop(offer.side, offer.player_id, stat, float(offer.line)),
                float(offer.over_american),
            )
            ranked_rows.append(
                {
                    "game": game,
                    "team": offer.team,
                    "player_name": offer.player_name,
                    "stat": label,
                    "min": float(box.teams[offer.side].stats["MIN"][:, index].mean()),
                    "mean": float(column.mean()),
                    "line": float(offer.line),
                    "odds": int(offer.over_american),
                    "prop_over": priced.p_over,
                    "over_ev": priced.ev,
                    "side": offer.side,
                    "player_id": offer.player_id,
                    "box_stat": stat,
                }
            )
    ranked = pd.DataFrame(ranked_rows)
    if ranked.empty:
        print(f"{game}: no priced props")
        return {"priced": ranked, "pairs": ranked, "portfolio": None}

    positive = ranked.loc[ranked["over_ev"] > 0].reset_index(drop=True)
    pair_rows = []
    for left, right in combinations(range(len(positive)), 2):
        a = positive.iloc[left]
        b = positive.iloc[right]
        priced = prop_probability(
            box,
            [
                Prop(a.side, a.player_id, a.box_stat, float(a.line)),
                Prop(b.side, b.player_id, b.box_stat, float(b.line)),
            ],
        )
        if a.player_id == b.player_id:
            link = "same player"
        elif a.team == b.team:
            link = "teammates"
        else:
            link = "opponents"
        pair_rows.append(
            {
                "game": game,
                "link": link,
                "a": f"{a.player_name} {a.stat} {a.line:g}",
                "b": f"{b.player_name} {b.stat} {b.line:g}",
                "p_a": priced.marginals[0],
                "p_b": priced.marginals[1],
                "joint": priced.joint,
                "product": priced.independent_product,
                "gap": priced.joint - priced.independent_product,
            }
        )
    top = ranked.sort_values("over_ev", ascending=False).head(5)
    legs = [
        Prop(row.side, row.player_id, row.box_stat, float(row.line))
        for row in top.itertuples(index=False)
    ]
    held = portfolio_risk(box, legs, top["odds"].tolist())
    names_book = " | ".join(
        f"{row.player_name} {row.stat} {row.line:g}" for row in top.itertuples(index=False)
    )
    portfolio = {
        "game": game,
        "n": held.n,
        "legs": names_book,
        "expected_profit": held.expected_profit,
        "risk": held.risk,
        "p_loss": held.p_loss,
        "joint": held.joint,
        "product": held.independent_product,
    }
    print(
        f"{game}: {len(ranked)} lines, {len(positive)} with over_ev > 0, "
        f"top over_ev {top['over_ev'].iloc[0]:.3f}"
    )
    visible = ranked.drop(columns=["side", "player_id", "box_stat"])
    return {
        "priced": visible,
        "pairs": pd.DataFrame(pair_rows),
        "portfolio": portfolio,
    }


In [6]:
priced_frames = []
pair_frames = []
portfolio_rows = []
stopped = []
for listed in games:
    try:
        slate = player_slate(board, posted, roster, listed["home"], outs, probable)
    except (ValueError, AssertionError) as exc:
        print(f"{listed['away']} @ {listed['home']} stopped: {exc}")
        stopped.append({"game": f"{listed['away']} @ {listed['home']}", "error": str(exc)})
        continue
    slate["date"] = slate_date
    try:
        result = run_game(slate)
    except (ValueError, AssertionError) as exc:
        print(f"{listed['away']} @ {listed['home']} stopped: {exc}")
        stopped.append({"game": f"{listed['away']} @ {listed['home']}", "error": str(exc)})
        continue
    if not result["priced"].empty:
        priced_frames.append(result["priced"])
    if not result["pairs"].empty:
        pair_frames.append(result["pairs"])
    if result["portfolio"] is not None:
        portfolio_rows.append(result["portfolio"])

priced = pd.concat(priced_frames, ignore_index=True) if priced_frames else pd.DataFrame()
pairs = pd.concat(pair_frames, ignore_index=True) if pair_frames else pd.DataFrame()
portfolios = pd.DataFrame(portfolio_rows)
print(
    f"{len(games) - len(stopped)} games priced, {len(stopped)} stopped, "
    f"{len(priced)} lines"
)
if stopped:
    display(pd.DataFrame(stopped))



BOS at MIL | total 222.5 | team spread 12.5
               mean     sd     p05     p25     p50     p75     p95
pace          98.40   3.99   91.78   95.84   98.45  100.97  104.93
team_poss     98.85   4.71   91.89   95.89   98.66  101.30  106.47
team_pts     105.06  11.44   86.44   97.41  105.05  112.72  123.86
opp_pts      117.52  11.42   98.88  109.93  117.37  125.07  136.37
game_total   222.58  18.05  193.04  210.72  222.35  234.45  252.56
team_margin  -12.46  14.04  -35.44  -22.02  -12.28   -3.09   10.60
P(OT): 4.0%


/Users/alexgonzalez/Documents/nba_quant_analysis/nba_game_simulator/src/game_opportunities/train.py:180: UserWarning: zero-minute rows have positive statistics; per-minute denominators exclude them
  prepared = _prepare_opportunity_logs(logs)


BOS @ MIL: 78 lines, 8 with over_ev > 0, top over_ev 0.121

BKN at DAL | total 224.0 | team spread -7.5
               mean     sd     p05     p25     p50     p75     p95
pace         100.12   3.94   93.59   97.52  100.13  102.67  106.68
team_poss    100.73   4.86   93.63   97.73  100.41  103.19  108.76
team_pts     115.74  11.35   96.90  108.14  115.85  123.34  134.28
opp_pts      108.51  11.34   89.94  100.93  108.54  116.10  127.12
game_total   224.26  17.95  195.05  212.14  224.24  236.27  253.67
team_margin    7.23  13.88  -15.75   -2.17    7.39   16.46   30.00
P(OT): 5.1%


/Users/alexgonzalez/Documents/nba_quant_analysis/nba_game_simulator/src/game_opportunities/train.py:180: UserWarning: zero-minute rows have positive statistics; per-minute denominators exclude them
  prepared = _prepare_opportunity_logs(logs)


BKN @ DAL: 63 lines, 16 with over_ev > 0, top over_ev 0.599

LAL at DET | total 217.0 | team spread -7.0
               mean     sd     p05     p25     p50     p75     p95
pace          97.63   4.02   91.08   94.97   97.64  100.24  104.27
team_poss     98.19   4.84   91.12   95.15   97.85  100.75  106.33
team_pts     112.01  11.56   92.90  104.28  112.08  119.74  130.98
opp_pts      105.04  11.48   85.93   97.39  105.26  112.56  123.91
game_total   217.05  18.18  186.84  205.10  217.02  229.06  247.42
team_margin    6.97  14.15  -16.15   -2.60    7.02   16.35   30.29
P(OT): 5.0%


/Users/alexgonzalez/Documents/nba_quant_analysis/nba_game_simulator/src/game_opportunities/train.py:180: UserWarning: zero-minute rows have positive statistics; per-minute denominators exclude them
  prepared = _prepare_opportunity_logs(logs)


... (5 outputs left out)

/Users/alexgonzalez/Documents/nba_quant_analysis/nba_game_simulator/src/game_opportunities/train.py:180: UserWarning: zero-minute rows have positive statistics; per-minute denominators exclude them
  prepared = _prepare_opportunity_logs(logs)


PHI @ CHI: no underdog lines matched
PHI @ CHI: no priced props

UTA at MIN | total 223.0 | team spread -12.0
               mean     sd     p05     p25     p50     p75     p95
pace         102.64   3.96   96.16  100.08  102.64  105.21  109.10
team_poss    103.09   4.71   96.13  100.17  102.86  105.56  110.80
team_pts     117.50  11.29   98.90  109.88  117.52  125.03  136.13
opp_pts      105.29  11.49   86.73   97.52  105.27  112.90  124.59
game_total   222.79  18.07  193.22  210.78  222.76  234.65  252.78
team_margin   12.21  13.87  -10.82    2.92   12.26   21.62   34.71
P(OT): 3.8%


/Users/alexgonzalez/Documents/nba_quant_analysis/nba_game_simulator/src/game_opportunities/train.py:180: UserWarning: zero-minute rows have positive statistics; per-minute denominators exclude them
  prepared = _prepare_opportunity_logs(logs)


UTA @ MIN: 62 lines, 19 with over_ev > 0, top over_ev 0.323
7 games priced, 0 stopped, 403 lines


## Prop probabilities

`prop_over` is the share of that game's rows strictly above the line. `mean` is the average of the stat on those rows. A combo is the sum of its pieces on the same row.


In [7]:
def show(frame, digits):
    shown = frame.copy()
    for column, places in digits.items():
        if column in shown.columns:
            shown[column] = shown[column].round(places)
    display(shown.reset_index(drop=True))


if priced.empty:
    print("no priced props")
else:
    print(f"{len(priced)} overs on {slate_date}, {market_audit['book']}")
    for game, frame in priced.groupby("game", sort=False):
        print(game)
        show(
            frame.drop(columns="game").sort_values("prop_over", ascending=False),
            {"min": 2, "mean": 2, "prop_over": 3, "over_ev": 3},
        )


403 overs on 2026-11-08, underdog
BOS @ MIL


,team,player_name,stat,min,mean,line,odds,prop_over,over_ev
0,MIL,Kyle Kuzma,r+a,27.66,8.01,6.5,-148,0.626,0.050
1,MIL,Tyler Herro,stl,30.09,1.03,0.5,-179,0.620,-0.034
2,BOS,Paul George,reb,29.80,5.53,4.5,-125,0.581,0.045
3,MIL,Kyle Kuzma,p+r+a,27.66,18.13,16.5,-118,0.566,0.045
4,BOS,Payton Pritchard,reb,22.87,3.12,2.5,-159,0.560,-0.088
...,...,...,...,...,...,...,...,...,...
73,MIL,Kel'el Ware,r+a,15.50,5.17,7.5,101,0.248,-0.501
74,MIL,Kel'el Ware,reb,15.50,4.26,6.5,-115,0.230,-0.571
75,MIL,AJ Green,stl,17.36,0.27,0.5,181,0.222,-0.377
76,BOS,Mitchell Robinson,reb,13.18,3.43,5.5,-112,0.214,-0.595


BKN @ DAL


,team,player_name,stat,min,mean,line,odds,prop_over,over_ev
0,DAL,P.J. Washington,p+r+a,33.05,23.95,21.5,-108,0.607,0.168
1,BKN,Nolan Traore,reb,26.76,3.45,2.5,164,0.606,0.599
2,DAL,Daniel Gafford,pts,22.68,8.97,7.5,-130,0.585,0.036
3,BKN,Danny Wolf,p+r+a,25.80,19.60,17.5,102,0.582,0.175
4,DAL,Kyrie Irving,reb,29.09,4.35,3.5,-136,0.574,-0.003
...,...,...,...,...,...,...,...,...,...
58,DAL,Cooper Flagg,p+a,30.86,25.00,30.5,-118,0.227,-0.580
59,DAL,Walter Clayton Jr.,ast,11.08,1.86,3.5,-104,0.201,-0.606
60,DAL,Cooper Flagg,p+r+a,30.86,31.10,38.5,-103,0.198,-0.610
61,DAL,Cooper Flagg,ast,30.86,4.62,6.5,126,0.195,-0.559


LAL @ DET


,team,player_name,stat,min,mean,line,odds,prop_over,over_ev
0,LAL,Ziaire Williams,p+r+a,28.83,19.24,13.5,-103,0.782,0.541
1,LAL,Quentin Grimes,reb,34.08,4.63,2.5,-154,0.754,0.244
2,LAL,Ziaire Williams,pts,28.83,12.72,8.5,-120,0.745,0.365
3,LAL,Ziaire Williams,reb,28.83,3.95,2.5,-141,0.678,0.160
4,LAL,Quentin Grimes,ast,34.08,3.60,2.5,-159,0.676,0.101
...,...,...,...,...,...,...,...,...,...
60,LAL,Jake LaRavia,reb,15.17,2.74,4.5,-113,0.224,-0.578
61,DET,Cade Cunningham,p+r+a,34.00,32.62,39.5,-110,0.220,-0.581
62,LAL,Austin Reaves,ast,36.75,4.37,6.5,-114,0.164,-0.693
63,DET,Cade Cunningham,p+a,34.00,26.75,34.5,-107,0.164,-0.684


... (2 outputs left out)

NOP @ LAC


,team,player_name,stat,min,mean,line,odds,prop_over,over_ev
0,LAC,Brook Lopez,p+r+a,31.03,25.53,19.5,-108,0.749,0.442
1,LAC,Brook Lopez,r+a,31.03,9.27,6.5,-122,0.721,0.312
2,LAC,Brook Lopez,pts,31.03,16.26,12.5,-103,0.694,0.368
3,NOP,Jeremiah Fears,reb,28.16,4.87,3.5,-164,0.651,0.048
4,LAC,Brook Lopez,reb,31.03,7.39,5.5,118,0.649,0.415
...,...,...,...,...,...,...,...,...,...
63,LAC,Max Strus,ast,13.51,1.50,2.5,110,0.253,-0.468
64,NOP,Zion Williamson,p+r+a,30.34,24.59,29.5,-104,0.247,-0.516
65,NOP,Dejounte Murray,p+r+a,29.91,22.90,28.5,-112,0.216,-0.592
66,NOP,Dejounte Murray,p+a,29.91,17.70,22.5,-115,0.213,-0.601


UTA @ MIN


,team,player_name,stat,min,mean,line,odds,prop_over,over_ev
0,UTA,Darryn Peterson,stl,29.97,1.34,0.5,-239,0.710,0.007
1,MIN,Jaden McDaniels,ast,34.70,3.73,2.5,-170,0.706,0.121
2,MIN,Ayo Dosunmu,reb,35.36,6.21,4.5,-145,0.662,0.119
3,MIN,Jaden McDaniels,r+a,34.70,10.49,8.5,-125,0.657,0.183
4,UTA,Jaxson Hayes,pts,28.24,10.68,8.5,-122,0.648,0.179
...,...,...,...,...,...,...,...,...,...
57,MIN,LaMelo Ball,ast,29.17,4.93,6.5,-145,0.243,-0.590
58,UTA,Keyonte George,p+a,30.65,22.05,27.5,-115,0.227,-0.575
59,MIN,Jaylen Clark,reb,8.64,1.37,2.5,120,0.225,-0.505
60,MIN,LaMelo Ball,p+a,29.17,22.58,28.5,-121,0.215,-0.607


## Correlations

Each row is two overs with positive expected value in the same game. `joint` is the share of rows where both clear. `product` multiplies the two separate chances. `gap` is joint minus product. The top of a game is the parlay. The bottom is the pair of singles.


In [8]:
if pairs.empty:
    print("no positive-EV pairs")
else:
    print(f"{len(pairs)} positive-EV pairs on {slate_date}")
    for game, frame in pairs.groupby("game", sort=False):
        ordered = frame.sort_values("gap", ascending=False)
        shown = ordered if len(ordered) <= 16 else pd.concat([ordered.head(8), ordered.tail(8)])
        print(f"{game}  {len(ordered)} pairs, showing {len(shown)}")
        show(
            shown.drop(columns="game"),
            {"p_a": 3, "p_b": 3, "joint": 3, "product": 3, "gap": 3},
        )


781 positive-EV pairs on 2026-11-08
BOS @ MIL  28 pairs, showing 16


,link,a,b,p_a,p_b,joint,product,gap
0,same player,Kyle Kuzma reb 4.5,Kyle Kuzma r+a 6.5,0.525,0.626,0.491,0.329,0.162
1,same player,Kyle Kuzma p+r+a 16.5,Kyle Kuzma r+a 6.5,0.566,0.626,0.463,0.354,0.109
2,same player,Kyle Kuzma ast 2.5,Kyle Kuzma r+a 6.5,0.547,0.626,0.445,0.343,0.102
3,same player,Kyle Kuzma reb 4.5,Kyle Kuzma p+r+a 16.5,0.525,0.566,0.393,0.297,0.096
4,same player,Kyle Kuzma ast 2.5,Kyle Kuzma p+r+a 16.5,0.547,0.566,0.378,0.309,0.069
5,teammates,Kyle Kuzma reb 4.5,Tyler Herro reb 4.5,0.525,0.514,0.314,0.270,0.044
6,teammates,Tyler Herro reb 4.5,Kyle Kuzma r+a 6.5,0.514,0.626,0.354,0.322,0.032
7,teammates,Tyler Herro reb 4.5,Kyle Kuzma p+r+a 16.5,0.514,0.566,0.314,0.291,0.023
8,teammates,Kyle Kuzma ast 2.5,Brayden Burries p+r+a 18.5,0.547,0.527,0.296,0.288,0.008
9,teammates,Jaime Jaquez Jr. pts 14.5,Brayden Burries p+r+a 18.5,0.516,0.527,0.280,0.272,0.008


BKN @ DAL  120 pairs, showing 16


,link,a,b,p_a,p_b,joint,product,gap
0,same player,Nolan Traore pts 13.5,Nolan Traore p+r+a 19.5,0.497,0.520,0.443,0.259,0.184
1,same player,P.J. Washington pts 13.5,P.J. Washington p+r+a 21.5,0.570,0.607,0.506,0.346,0.160
2,same player,Danny Wolf pts 9.5,Danny Wolf p+r+a 17.5,0.556,0.582,0.480,0.323,0.156
3,same player,Zaccharie Risacher reb 3.5,Zaccharie Risacher p+r+a 11.5,0.501,0.550,0.401,0.276,0.126
4,same player,Danny Wolf reb 5.5,Danny Wolf p+r+a 17.5,0.514,0.582,0.411,0.299,0.112
5,same player,P.J. Washington p+r+a 21.5,P.J. Washington r+a 8.5,0.607,0.493,0.397,0.299,0.098
6,same player,Nolan Traore reb 2.5,Nolan Traore p+r+a 19.5,0.606,0.520,0.388,0.315,0.074
7,same player,Danny Wolf ast 2.5,Danny Wolf p+r+a 17.5,0.464,0.582,0.340,0.270,0.071
8,teammates,Danny Wolf pts 9.5,Nolan Traore reb 2.5,0.556,0.606,0.331,0.337,-0.005
9,teammates,Danny Wolf pts 9.5,Nolan Traore pts 13.5,0.556,0.497,0.271,0.277,-0.006


LAL @ DET  153 pairs, showing 16


,link,a,b,p_a,p_b,joint,product,gap
0,same player,Duncan Robinson pts 15.5,Duncan Robinson p+r+a 20.5,0.583,0.593,0.535,0.346,0.189
1,same player,Matisse Thybulle pts 7.5,Matisse Thybulle p+r+a 11.5,0.596,0.646,0.553,0.385,0.168
2,same player,Quentin Grimes pts 11.5,Quentin Grimes p+r+a 17.5,0.526,0.646,0.495,0.340,0.155
3,same player,Ziaire Williams pts 8.5,Ziaire Williams p+r+a 13.5,0.745,0.782,0.707,0.582,0.125
4,same player,Quentin Grimes ast 2.5,Quentin Grimes p+r+a 17.5,0.676,0.646,0.507,0.437,0.071
5,same player,Quentin Grimes reb 2.5,Quentin Grimes p+r+a 17.5,0.754,0.646,0.554,0.487,0.067
6,same player,Ziaire Williams reb 2.5,Ziaire Williams p+r+a 13.5,0.678,0.782,0.588,0.530,0.058
7,same player,Duncan Robinson reb 2.5,Duncan Robinson p+r+a 20.5,0.520,0.593,0.365,0.309,0.056
8,teammates,Cade Cunningham reb 5.5,Paul Reed ast 2.5,0.494,0.468,0.227,0.231,-0.004
9,opponents,Matisse Thybulle pts 7.5,Duncan Robinson reb 2.5,0.596,0.520,0.306,0.310,-0.004


... (2 outputs left out)

NOP @ LAC  231 pairs, showing 16


,link,a,b,p_a,p_b,joint,product,gap
0,same player,Karlo Matković pts 7.5,Karlo Matković p+r+a 13.5,0.606,0.582,0.517,0.352,0.165
1,same player,Brook Lopez reb 5.5,Brook Lopez r+a 6.5,0.649,0.721,0.631,0.468,0.163
2,same player,Kris Dunn pts 8.5,Kris Dunn p+r+a 17.5,0.579,0.580,0.487,0.336,0.150
3,same player,Jeremiah Fears reb 3.5,Jeremiah Fears r+a 7.5,0.651,0.575,0.513,0.374,0.139
4,same player,Brook Lopez pts 12.5,Brook Lopez p+r+a 19.5,0.694,0.749,0.655,0.520,0.136
5,same player,Jeremiah Fears ast 3.5,Jeremiah Fears r+a 7.5,0.485,0.575,0.391,0.279,0.112
6,same player,Jeremiah Fears p+r+a 22.5,Jeremiah Fears r+a 7.5,0.520,0.575,0.399,0.299,0.100
7,same player,Kris Dunn reb 4.5,Kris Dunn p+r+a 17.5,0.535,0.580,0.405,0.311,0.094
8,opponents,Darius Garland reb 2.5,Jeremiah Fears ast 3.5,0.581,0.485,0.278,0.282,-0.003
9,opponents,Karlo Matković pts 7.5,Rui Hachimura reb 5.5,0.606,0.598,0.359,0.362,-0.003


UTA @ MIN  171 pairs, showing 16


,link,a,b,p_a,p_b,joint,product,gap
0,same player,Jaden McDaniels p+a 20.5,Jaden McDaniels p+r+a 26.5,0.537,0.571,0.486,0.306,0.179
1,same player,Jonathan Kuminga reb 6.5,Jonathan Kuminga r+a 8.5,0.452,0.601,0.431,0.272,0.159
2,same player,Jaden McDaniels reb 5.5,Jaden McDaniels r+a 8.5,0.603,0.657,0.548,0.396,0.151
3,same player,Ayo Dosunmu reb 4.5,Ayo Dosunmu r+a 9.5,0.662,0.576,0.524,0.382,0.143
4,same player,Jonathan Kuminga p+r+a 23.5,Jonathan Kuminga r+a 8.5,0.539,0.601,0.428,0.324,0.104
5,same player,Jonathan Kuminga reb 6.5,Jonathan Kuminga p+r+a 23.5,0.452,0.539,0.342,0.244,0.098
6,same player,Jaden McDaniels p+r+a 26.5,Jaden McDaniels r+a 8.5,0.571,0.657,0.468,0.375,0.093
7,same player,Jonathan Kuminga ast 2.5,Jonathan Kuminga r+a 8.5,0.642,0.601,0.472,0.386,0.086
8,opponents,Ayo Dosunmu reb 4.5,Darryn Peterson stl 0.5,0.662,0.710,0.466,0.470,-0.004
9,teammates,Jaden McDaniels ast 2.5,Terrence Shannon Jr p+r+a 16.5,0.706,0.562,0.392,0.397,-0.005


## Expected value

`over_ev` is the profit per dollar on that over. A push returns the stake. These are the highest overs on the slate, across games.


In [9]:
if priced.empty:
    print("no priced props")
else:
    top_day = priced.sort_values("over_ev", ascending=False).head(15)
    print(f"top {len(top_day)} overs by expected value on {slate_date}")
    show(top_day, {"min": 2, "mean": 2, "prop_over": 3, "over_ev": 3})


top 15 overs by expected value on 2026-11-08


,game,team,player_name,stat,min,mean,line,odds,prop_over,over_ev
0,BKN @ DAL,BKN,Nolan Traore,reb,26.76,3.45,2.5,164,0.606,0.599
1,LAL @ DET,LAL,Ziaire Williams,p+r+a,28.83,19.24,13.5,-103,0.782,0.541
2,NOP @ LAC,LAC,Brook Lopez,p+r+a,31.03,25.53,19.5,-108,0.749,0.442
3,NOP @ LAC,LAC,Brook Lopez,reb,31.03,7.39,5.5,118,0.649,0.415
4,NOP @ LAC,LAC,Brook Lopez,pts,31.03,16.26,12.5,-103,0.694,0.368
5,LAL @ DET,LAL,Ziaire Williams,pts,28.83,12.72,8.5,-120,0.745,0.365
6,MIA @ SAC,SAC,Darius Acuff Jr.,reb,32.74,4.17,3.5,146,0.545,0.340
7,UTA @ MIN,MIN,Jonathan Kuminga,ast,32.47,3.38,2.5,106,0.642,0.323
8,NOP @ LAC,LAC,Brook Lopez,r+a,31.03,9.27,6.5,-122,0.721,0.312
9,MIA @ SAC,MIA,Davion Mitchell,pts,35.65,16.44,13.5,-102,0.658,0.304


## Portfolio risk

Each game stakes one dollar on its five highest overs. `expected_profit` and `risk` are the mean and standard deviation of the combined profit on that game's rows. `joint` is the share of rows where all five clear. The day book adds the game portfolios. The games are separate scripts, so the day's risk is the square root of the summed variances.


In [10]:
if portfolios.empty:
    print("no portfolios")
else:
    print(f"one portfolio per game, $1 on each of its top overs, {market_audit['book']}")
    show(
        portfolios,
        {
            "expected_profit": 3,
            "risk": 3,
            "p_loss": 3,
            "joint": 3,
            "product": 3,
        },
    )
    day_profit = float(portfolios["expected_profit"].sum())
    day_risk = float(np.sqrt((portfolios["risk"] ** 2).sum()))
    print(
        f"day book of these {len(portfolios)} portfolios. "
        f"expected profit = {day_profit:.3f}; risk = {day_risk:.3f}"
    )


one portfolio per game, $1 on each of its top overs, underdog


,game,n,legs,expected_profit,risk,p_loss,joint,product
0,BOS @ MIL,5,Tyler Herro reb 4.5 | Kyle Kuzma ast 2.5 | Jai...,0.401,2.805,0.418,0.106,0.048
1,BKN @ DAL,5,Nolan Traore reb 2.5 | Nolan Traore stl 1.5 | ...,1.297,2.793,0.338,0.099,0.060
2,LAL @ DET,5,Ziaire Williams p+r+a 13.5 | Ziaire Williams p...,1.576,2.296,0.198,0.244,0.169
3,MIA @ SAC,5,Darius Acuff Jr. reb 3.5 | Davion Mitchell pts...,1.282,2.776,0.322,0.186,0.092
4,NOP @ LAC,5,Brook Lopez p+r+a 19.5 | Brook Lopez reb 5.5 |...,1.831,3.013,0.260,0.323,0.153
5,UTA @ MIN,5,Jonathan Kuminga ast 2.5 | Jonathan Kuminga r+...,1.048,2.328,0.287,0.150,0.104


day book of these 6 portfolios. expected profit = 7.434; risk = 6.568
